# 📘 Tutoriel : Offre Théorique de Transport — Datalake → MinIO → Visualisation

Ce notebook vous guide pas à pas dans le workflow suivant :

1. **Charger** les données d'offre théorique depuis le Datalake Azure (format Parquet)
2. **Filtrer et transformer** les données (période, mode de transport, colonnes utiles)
3. **Sauvegarder** le résultat filtré dans un bucket MinIO (S3-compatible)
4. **Recharger** depuis MinIO et **visualiser** le dataset pour en comprendre la structure

> **Dataset** : `offre_theorique_consolide.parquet` — il contient les horaires théoriques de tous les arrêts de transport en Île-de-France (bus, métro, tramway, etc.).

In [1]:
!pip install pandas pyarrow fsspec adlfs boto3 plotly -q

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import io
import boto3

## 1️⃣ Chargement depuis le Datalake Azure

On lit directement un fichier Parquet hébergé sur le Datalake Azure (`abfs://`).  
Le paramètre `storage_options={"anon": False}` utilise l'authentification Azure par défaut (Azure CLI / Managed Identity).

**Paramètres à ajuster :**
- `FILTERS` : période souhaitée (format `YYYY-MM-DD`)
- Le chemin pointe vers le gold de l'offre théorique consolidée

In [3]:
# --- Configuration du chemin Datalake ---
DATALAKE = "abfs://dppdatalakeprddls01.dfs.core.windows.net/"
DOMAIN = "offre-de-transport/"
SOURCE = "prim/"
DATA = "offre-theorique/"
QUALITY = "gold/"
DATASET = "offre_theorique_consolide.parquet"

FULL_PATH = f"{DATALAKE}{DOMAIN}{SOURCE}{DATA}{QUALITY}{DATASET}"

# --- Filtre sur la période (2 semaines de juin 2025) ---
FILTERS = [("date", ">=", "2025-06-10"), ("date", "<=", "2025-06-23")]

# --- Lecture depuis le Datalake ---
df_raw = pd.read_parquet(
    FULL_PATH,
    storage_options={"anon": False},
    filters=FILTERS,
)
df_raw["date"] = pd.to_datetime(df_raw['date'])

print(f"✅ Données chargées : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes")
print(f"📅 Période : {df_raw['date'].min()} → {df_raw['date'].max()}")
df_raw.head()

✅ Données chargées : 38,179,383 lignes × 18 colonnes
📅 Période : 2025-06-10 00:00:00 → 2025-06-23 00:00:00


,route_id,agency_id,trip_id,trip_headsign,direction_id,stop_id,departure_time,arrival_time,stop_sequence,route_long_name,route_short_name,parent_station,route_type,stop_name,stop_lat,stop_lon,zone_stop_id,date
0,C00852,78,5461464,Gare de Savigny Nandy,0,491822,2025-06-10 05:03:00+00:00,2025-06-10 05:03:00+00:00,0,3737,3737,62290.0,3,Gare de Savigny-Nandy,48.595643,2.584667,473550,2025-06-10
1,C00852,78,5461464,Gare de Savigny Nandy,0,33191,2025-06-10 05:05:00+00:00,2025-06-10 05:05:00+00:00,1,3737,3737,62271.0,3,René Cassin,48.589857,2.584035,53532,2025-06-10
2,C00852,78,5461464,Gare de Savigny Nandy,0,33189,2025-06-10 05:06:00+00:00,2025-06-10 05:06:00+00:00,2,3737,3737,62264.0,3,Droits de l'Homme,48.587496,2.585545,53540,2025-06-10
3,C00852,78,5461464,Gare de Savigny Nandy,0,33187,2025-06-10 05:07:00+00:00,2025-06-10 05:07:00+00:00,3,3737,3737,62264.0,3,Victor Schoelcher,48.586216,2.584796,53566,2025-06-10
4,C00852,78,5461464,Gare de Savigny Nandy,0,33185,2025-06-10 05:08:00+00:00,2025-06-10 05:08:00+00:00,4,3737,3737,62257.0,3,Collège la Grange du Bois,48.585450,2.581928,53538,2025-06-10


## 2️⃣ Exploration rapide du dataset brut

Avant de filtrer, regardons la structure des données :
- **18 colonnes** : identifiants de ligne/arrêt, horaires, coordonnées GPS, type de transport…
- **`route_type`** : `0` = Tramway, `1` = Métro, `2` = Train, `3` = Bus
- **`direction_id`** : `0` = Aller, `1` = Retour

In [4]:
# Aperçu des types et valeurs manquantes
print("--- Types des colonnes ---")
print(df_raw.dtypes.to_string())
print("\n--- Valeurs manquantes ---")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0].to_string())
print("\n--- Statistiques clés ---")
print(f"  Nb dates          : {df_raw['date'].nunique()}")
print(f"  Nb lignes (routes): {df_raw['route_id'].nunique()}")
print(f"  Nb arrêts (stops) : {df_raw['stop_id'].nunique()}")
print(f"  Nb opérateurs     : {df_raw['agency_id'].nunique()}")

# Mapping des types de transport
ROUTE_TYPE_MAP = {0: "Tramway", 1: "Métro", 2: "Train", 3: "Bus"}
print("\n--- Répartition par mode de transport ---")
print(df_raw["route_type"].map(ROUTE_TYPE_MAP).value_counts().to_string())

--- Types des colonnes ---
route_id                         object
agency_id                        object
trip_id                           int64
trip_headsign                    object
direction_id                       int8
stop_id                          object
departure_time      datetime64[ns, UTC]
arrival_time        datetime64[ns, UTC]
stop_sequence                     int16
route_long_name                  object
route_short_name                 object
parent_station                  float64
route_type                         int8
stop_name                        object
stop_lat                        float64
stop_lon                        float64
zone_stop_id                     object
date                     datetime64[ns]

--- Valeurs manquantes ---
Series([], )

--- Statistiques clés ---
  Nb dates          : 14
  Nb lignes (routes): 1990
  Nb arrêts (stops) : 35578
  Nb opérateurs     : 63

--- Répartition par mode de transport ---
route_type
Bus        32111077
Métro 

## 3️⃣ Filtrage et transformation des données

On va maintenant retravailler le dataset :
1. **Filtrer** uniquement les **bus** (`route_type == 3`) — le mode le plus représenté
2. **Ne garder qu'une semaine** (10–16 juin 2025) pour alléger le fichier
3. **Ajouter des colonnes dérivées** : jour de la semaine, heure de départ, tranche horaire
4. **Sélectionner** les colonnes utiles et renommer pour plus de clarté

In [5]:
# --- 3a. Filtre sur le mode Bus et la 1ère semaine ---
df = df_raw[
    (df_raw["route_type"] == 3) &
    (df_raw["date"] >= "2025-06-10") &
    (df_raw["date"] <= "2025-06-16")
].copy()

print(f"🚌 Après filtre Bus + 1 semaine : {df.shape[0]:,} lignes (vs {df_raw.shape[0]:,} avant)")

# --- 3b. Colonnes dérivées ---
# Jour de la semaine (lundi=0, dimanche=6)
df["date_dt"] = pd.to_datetime(df["date"])
df["jour_semaine"] = df["date_dt"].dt.day_name()

# Heure de départ (entière, 0-23)
df["heure_depart"] = df["departure_time"].dt.hour

# Tranche horaire (pointe matin, creux, pointe soir, soirée)
def tranche_horaire(h):
    if 6 <= h < 10:
        return "Pointe matin (6h-10h)"
    elif 10 <= h < 16:
        return "Creux (10h-16h)"
    elif 16 <= h < 20:
        return "Pointe soir (16h-20h)"
    else:
        return "Soirée/Nuit"

df["tranche_horaire"] = df["heure_depart"].apply(tranche_horaire)

# --- 3c. Sélection et renommage des colonnes utiles ---
COLONNES_FINALES = {
    "date": "date",
    "jour_semaine": "jour_semaine",
    "route_id": "ligne_id",
    "route_short_name": "ligne_nom",
    "agency_id": "operateur_id",
    "trip_id": "course_id",
    "trip_headsign": "destination",
    "direction_id": "direction",
    "stop_id": "arret_id",
    "stop_name": "arret_nom",
    "stop_lat": "latitude",
    "stop_lon": "longitude",
    "departure_time": "heure_depart_dt",
    "heure_depart": "heure_depart",
    "tranche_horaire": "tranche_horaire",
    "stop_sequence": "ordre_arret",
}

df_clean = df[list(COLONNES_FINALES.keys())].rename(columns=COLONNES_FINALES)

print(f"✅ Dataset nettoyé : {df_clean.shape[0]:,} lignes × {df_clean.shape[1]} colonnes")
df_clean.head()

🚌 Après filtre Bus + 1 semaine : 16,038,124 lignes (vs 38,179,383 avant)
✅ Dataset nettoyé : 16,038,124 lignes × 16 colonnes


,date,jour_semaine,ligne_id,ligne_nom,operateur_id,course_id,destination,direction,arret_id,arret_nom,latitude,longitude,heure_depart_dt,heure_depart,tranche_horaire,ordre_arret
0,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,491822,Gare de Savigny-Nandy,48.595643,2.584667,2025-06-10 05:03:00+00:00,5,Soirée/Nuit,0
1,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33191,René Cassin,48.589857,2.584035,2025-06-10 05:05:00+00:00,5,Soirée/Nuit,1
2,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33189,Droits de l'Homme,48.587496,2.585545,2025-06-10 05:06:00+00:00,5,Soirée/Nuit,2
3,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33187,Victor Schoelcher,48.586216,2.584796,2025-06-10 05:07:00+00:00,5,Soirée/Nuit,3
4,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33185,Collège la Grange du Bois,48.585450,2.581928,2025-06-10 05:08:00+00:00,5,Soirée/Nuit,4


## 4️⃣ Sauvegarde dans le MinIO (S3-compatible)

Le MinIO est un stockage objet compatible S3, utilisé sur la plateforme Onyxia.  
On va sauvegarder notre dataset filtré au format Parquet directement dans un bucket.

> ⚠️ **Remplacez** les valeurs `ACCESS_KEY`, `SECRET_KEY` et `SESSION_TOKEN` par vos identifiants Onyxia.  
> Vous les trouverez dans l'interface Onyxia → **Mon compte** → **Connexion au stockage**.

In [ ]:
# --- Configuration MinIO ---
ACCESS_KEY = "your-access-key"          # ← À remplacer
SECRET_KEY = "your-secret-key"          # ← À remplacer
SESSION_TOKEN = "your-session-token"    # ← À remplacer
REGION = "fr-central"
ENDPOINT_URL = "https://minio.data-platform-self-service.net"

BUCKET = "dlb-hackathon"
OBJECT_KEY = "datasets-diffusion/2025/offre_bus_1semaine_202506.parquet"

# --- Connexion au client S3 ---
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    aws_session_token=SESSION_TOKEN,
    region_name=REGION,
)

# --- Upload du DataFrame en Parquet directement en mémoire ---
buffer = io.BytesIO()
df_clean.to_parquet(buffer, engine="pyarrow", compression="snappy", index=False)
buffer.seek(0)

s3.put_object(Bucket=BUCKET, Key=OBJECT_KEY, Body=buffer.getvalue())

print(f"✅ Fichier uploadé dans MinIO : s3://{BUCKET}/{OBJECT_KEY}")
print(f"   Taille : {buffer.getbuffer().nbytes / 1024 / 1024:.1f} Mo")

## 5️⃣ Rechargement depuis MinIO

On va maintenant relire le fichier Parquet directement depuis le bucket MinIO, comme le ferait un autre utilisateur de la plateforme.

In [7]:
# --- Téléchargement depuis MinIO ---
response = s3.get_object(Bucket=BUCKET, Key=OBJECT_KEY)
df_minio = pd.read_parquet(io.BytesIO(response["Body"].read()))

print(f"✅ Fichier rechargé depuis MinIO : {df_minio.shape[0]:,} lignes × {df_minio.shape[1]} colonnes")
df_minio.head()

✅ Fichier rechargé depuis MinIO : 16,038,124 lignes × 16 colonnes


,date,jour_semaine,ligne_id,ligne_nom,operateur_id,course_id,destination,direction,arret_id,arret_nom,latitude,longitude,heure_depart_dt,heure_depart,tranche_horaire,ordre_arret
0,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,491822,Gare de Savigny-Nandy,48.595643,2.584667,2025-06-10 05:03:00+00:00,5,Soirée/Nuit,0
1,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33191,René Cassin,48.589857,2.584035,2025-06-10 05:05:00+00:00,5,Soirée/Nuit,1
2,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33189,Droits de l'Homme,48.587496,2.585545,2025-06-10 05:06:00+00:00,5,Soirée/Nuit,2
3,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33187,Victor Schoelcher,48.586216,2.584796,2025-06-10 05:07:00+00:00,5,Soirée/Nuit,3
4,2025-06-10,Tuesday,C00852,3737,78,5461464,Gare de Savigny Nandy,0,33185,Collège la Grange du Bois,48.585450,2.581928,2025-06-10 05:08:00+00:00,5,Soirée/Nuit,4


## 6️⃣ Visualisations pour comprendre le dataset

Maintenant que les données sont chargées, on va créer quelques graphiques pour comprendre l'offre de bus :

1. **Nombre de passages par jour** — Y a-t-il des variations jour par jour ?
2. **Distribution horaire** — À quelles heures les bus circulent-ils le plus ?
3. **Top 15 des lignes** — Quelles lignes ont le plus de passages ?
4. **Carte des arrêts** — Où sont situés les arrêts de bus en Île-de-France ?

### 📊 6a. Nombre de passages de bus par jour

In [8]:
# Nombre de passages par jour
passages_jour = df_minio.groupby("date").size().reset_index(name="nb_passages")

fig = px.bar(
    passages_jour,
    x="date",
    y="nb_passages",
    title="Nombre de passages de bus par jour",
    labels={"date": "Date", "nb_passages": "Nombre de passages"},
    color_discrete_sequence=["#1f77b4"],
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 📊 6b. Distribution horaire des départs

On s'attend à voir des **pics aux heures de pointe** (7h-9h et 17h-19h) et un creux la nuit.

In [9]:
# Distribution horaire des départs (agrégé sur toute la semaine)
passages_heure = df_minio.groupby("heure_depart").size().reset_index(name="nb_passages")

# Couleur par tranche horaire
def couleur_tranche(h):
    if 6 <= h < 10:
        return "Pointe matin"
    elif 10 <= h < 16:
        return "Creux"
    elif 16 <= h < 20:
        return "Pointe soir"
    else:
        return "Soirée/Nuit"

passages_heure["tranche"] = passages_heure["heure_depart"].apply(couleur_tranche)

fig = px.bar(
    passages_heure,
    x="heure_depart",
    y="nb_passages",
    color="tranche",
    title="Distribution horaire des départs de bus (semaine du 10–16 juin 2025)",
    labels={"heure_depart": "Heure", "nb_passages": "Nombre de passages", "tranche": "Tranche"},
    color_discrete_map={
        "Pointe matin": "#e74c3c",
        "Creux": "#f39c12",
        "Pointe soir": "#e74c3c",
        "Soirée/Nuit": "#7f8c8d",
    },
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

### 📊 6c. Top 15 des lignes de bus les plus fréquentes

In [10]:
# Top 15 des lignes par nombre de passages sur la semaine
top_lignes = (
    df_minio
    .groupby(["ligne_id", "ligne_nom"])
    .size()
    .reset_index(name="nb_passages")
    .sort_values("nb_passages", ascending=False)
    .head(15)
)

fig = px.bar(
    top_lignes,
    x="nb_passages",
    y="ligne_nom",
    orientation="h",
    title="Top 15 des lignes de bus — Nombre de passages sur la semaine",
    labels={"ligne_nom": "Ligne", "nb_passages": "Nombre de passages"},
    color_discrete_sequence=["#2ecc71"],
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### 📊 6d. Répartition des passages par tranche horaire et par jour

Un heatmap pour visualiser l'intensité de l'offre selon le jour et la tranche horaire.

In [11]:
# Heatmap : heure × jour
heatmap_data = (
    df_minio
    .groupby(["date", "heure_depart"])
    .size()
    .reset_index(name="nb_passages")
)

fig = px.density_heatmap(
    heatmap_data,
    x="heure_depart",
    y="date",
    z="nb_passages",
    title="Intensité de l'offre bus — Heure × Jour",
    labels={"heure_depart": "Heure", "date": "Date", "nb_passages": "Passages"},
    color_continuous_scale="YlOrRd",
)
fig.update_layout(xaxis=dict(dtick=1), yaxis=dict(autorange="reversed"))
fig.show()

### 🗺️ 6e. Carte des arrêts de bus

On utilise les coordonnées GPS (`latitude`, `longitude`) pour afficher les arrêts sur une carte.  
La taille des points est proportionnelle au nombre de passages à l'arrêt.

In [12]:
# Agréger par arrêt : position GPS + nombre de passages
arrets = (
    df_minio
    .groupby(["arret_id", "arret_nom", "latitude", "longitude"])
    .size()
    .reset_index(name="nb_passages")
)

# Supprimer les arrêts sans coordonnées
arrets = arrets.dropna(subset=["latitude", "longitude"])
print(f"📍 {arrets.shape[0]} arrêts avec coordonnées GPS")

# Carte avec Plotly (scatter_mapbox)
fig = px.scatter_map(
    arrets,
    lat="latitude",
    lon="longitude",
    size="nb_passages",
    color="nb_passages",
    hover_name="arret_nom",
    hover_data={"nb_passages": True, "latitude": ":.4f", "longitude": ":.4f"},
    title="Carte des arrêts de bus en Île-de-France (taille = nb de passages)",
    color_continuous_scale="Viridis",
    size_max=12,
    zoom=9,
    center={"lat": 48.86, "lon": 2.35},
    height=700,
)
fig.show()

📍 33615 arrêts avec coordonnées GPS


## ✅ Récapitulatif

Ce notebook vous a montré comment :

| Étape | Action | Outil |
|-------|--------|-------|
| 1 | Charger des données Parquet depuis le **Datalake Azure** | `pandas.read_parquet` + `abfs://` |
| 2 | Explorer le dataset (types, valeurs manquantes, distribution) | `pandas` |
| 3 | Filtrer (mode bus, 1 semaine) et enrichir (jour, tranche horaire) | `pandas` |
| 4 | Sauvegarder dans **MinIO** (S3-compatible) | `boto3.put_object` |
| 5 | Recharger depuis MinIO | `boto3.get_object` |
| 6 | Visualiser (barres, heatmap, carte) | `plotly` |

> 💡 **Pour aller plus loin** : croisez ces données d'offre théorique avec l'offre réalisée pour mesurer le taux de réalisation par ligne !

In [ ]:
OBJECT_KEY = "notebooks/etl_and_visualize.ipynb"

LOCAL_NOTEBOOK_PATH = "etl_and_visualize.ipynb"

# --- Connexion au client S3 ---
s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    aws_session_token=SESSION_TOKEN,
    region_name=REGION,
)

# --- Upload du notebook ---
with open(LOCAL_NOTEBOOK_PATH, "rb") as f:
    s3.put_object(
        Bucket=BUCKET,
        Key=OBJECT_KEY,
        Body=f
    )

print(f"✅ Notebook uploadé : s3://{BUCKET}/{OBJECT_KEY}")